# Numerical Computation of The Bayes Update Rule for Bayesian Adaptive Filtering

This notebook aims to implement through numerical integration the unidimensional bayesian update rule for general distributions in both the prior and the likelihood.

$$ f_m(\theta_{t,m} | y_{1:t}) \propto f_m(\theta_{t,m} | y_{1:t-1})\,f_{\zeta_m}\!\big(y_t - x_{t,m}\theta_{t,m}\big) $$

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from numba import njit
from numba_progress import ProgressBar

from filters import (
    # parameter dtypes
    NLMS_params, sKF_params, sKF_L_params, skf_int_params, skf_L_int_params,
    # signal / environment helpers
    autocorr_matrix_calc, autocorr_matrix_estimate, AR_settling_time, std_behavior,
    # algorithms
    NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, sKF_L_exact_algorithm,
    sKF_integral_algorithm, sKF_L_integral_algorithm,
    # monte carlo driver
    MC_Simulations_Modular_Variance,
)

## Framework

All filter implementations, parameter dtypes and Monte Carlo drivers live in [`filters.py`](filters.py).

## Simulations

In [ ]:
NR = 200
N = int(200)
L = 64
ho = np.sinc(np.linspace(0, 1, L))
ho = ho / np.linalg.norm(ho)
h0 = np.zeros(L)
var_x = 1
var_v = 1e-3
# AR = np.array([1.0, 0.0])
# AR = np.array([1.0, -0.6, 0.85])
AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

mu = 0.1
delta = 1e-3

# sKF
epsilon = 1e-4
var_eta = 1e-3
v_tilde_0 = 1e-3

Algorithms = [NLMS_algorithm, sKF_algorithm]
NLMS_Parameters = np.void(("NLMS", mu, delta), dtype=NLMS_params)
sKF_Parameters = np.void(("sKF", epsilon, var_eta, v_tilde_0), dtype=sKF_params)
Alg_Parameters = [NLMS_Parameters, sKF_Parameters]

### Autocorrelation Matrix Test

In [ ]:
# Test the autorregressive process
tau = AR_settling_time(AR)

signals = std_behavior(N, ho, var_x, var_v, AR, tau)
x = signals['x']
d = signals['d']

print(tau)

In [ ]:
Rxx_est = autocorr_matrix_estimate(x, M = L)
rxx_est = Rxx_est[0,:]

plt.stem(rxx_est)

In [ ]:
aux = autocorr_matrix_calc(AR, 1, M = len(AR) - 1)
var_w = var_x/aux[0,0]

Rxx = autocorr_matrix_calc(AR, var_w, M = L)
rxx = Rxx[0,:]

plt.stem(rxx[:])

In [ ]:
print(f"Cosine Similarity: {rxx @ rxx_est / (np.linalg.norm(rxx)*np.linalg.norm(rxx_est))}")
print(f"Normalized Distance: {np.linalg.norm(Rxx - Rxx_est)/np.linalg.norm(Rxx)}")

### Monte Carlo Simulations

In [ ]:
Rxx = autocorr_matrix_calc(AR, var_w, M=L)
eig = np.linalg.eigvals(Rxx)
chi = np.max(eig) / np.min(eig)
print(f"Eigenvalue spread: {chi:.4} \n")

#### Eigenvalue spread

In [ ]:
Rxx = autocorr_matrix_calc(AR, var_w, M = L)
eig = np.linalg.eigvals(Rxx)
chi = np.max(eig)/np.min(eig)
print(f'Eigenvalue spread: {chi:.4} \n')

#### Bayesian Numerical Filter

##### Gaussian Likelihood Simulations

In [ ]:
L = 3
ho = np.sinc(np.linspace(0,1.5,L))
ho = ho/np.linalg.norm(ho) # ground truth
h0 = np.zeros(L)
var_x = 1
var_v = 1e-3
#AR = np.array([1.0, 0.0])
AR = np.array([1.0, -0.6, 0.85])
#AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

Rxx_est = autocorr_matrix_estimate(x, M = L)
eig = np.linalg.eigvals(Rxx)
chi = np.max(eig)/np.min(eig)
print(f'Eigenvalue spread: {chi:.4} \n')

rxx_est = Rxx_est[0,:]

plt.figure(figsize=(8, 3))
plt.stem(rxx_est)
plt.show()

##### PRUEBA MANUAL


In [ ]:
NR = 1
N = int(200)

epsilon = 0.01
var_theta_0 = 2 # v_tilde_0
var_eta = 10*var_v
dx_factor = 1/25
min_std_deviations = 10
# NUEVO - LAPLACIAN
b_eta = 2

Algorithms = [sKF_integral_algorithm, sKF_algorithm, sKF_L_algorithm]

skf_parameters = np.void((f"sKF_integral",
                          epsilon,
                          var_theta_0,
                          var_eta,
                          dx_factor,
                          min_std_deviations), dtype=skf_int_params)

sKF_parameters_paper = np.void(("sKF_paper", epsilon, var_eta, var_theta_0), dtype=sKF_params)

sKF_L_parameters_paper = np.void(("sKF_L_paper", epsilon, b_eta, var_theta_0), dtype=sKF_L_params)


Alg_Parameters = [skf_parameters, sKF_parameters_paper, sKF_L_parameters_paper]

with ProgressBar(total=NR) as PBar:
  MC_measures = MC_Simulations_Modular_Variance(N, NR, ho, var_x, var_v, h0, Algorithms, Alg_Parameters, AR, PBar)



In [ ]:
plt.plot(10*np.log10(MC_measures["sKF_integral"]['Jex']))
plt.plot(10*np.log10(MC_measures["sKF_paper"]['Jex']))
plt.ylabel("EMSE (dB)")
plt.xlabel("Iterations")
plt.title("sKF integral Learning Curve")
plt.show()

In [ ]:
h_num   = MC_measures["sKF_integral"]['h']
#h_L_num   = MC_measures["sKF_L_paper"]['h']
h_paper = MC_measures["sKF_paper"]['h']
L = h_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(h_num[:, k],   color=c, linestyle='-',  label=f"$w_{k}$ numerical")
  plt.plot(h_paper[:, k], color=c, linestyle='--', label=f"$w_{k}$ paper")
  #plt.plot(h_L_num[:, k], color='k', linestyle='-.', label=f"$w_{k}$ paper Laplace")
  plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("w")
plt.xlabel("Iterations")
plt.title("sKF weights: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
# plt.savefig("figures/skf_weights.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#h_num   = MC_measures["sKF_integral"]['h']
h_L_num  = MC_measures["sKF_L_paper"]['h']
h_L_50_51   = MC_measures["sKF_L_paper"]['h']
#h_paper = MC_measures["sKF_paper"]['h']
L = h_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(h_num[:, k],   color=c, linestyle='-',  label=f"$w_{k}$ numerical")
  plt.plot(h_paper[:, k], color=c, linestyle='--', label=f"$w_{k}$ paper")
  plt.plot(h_L_50_51[:, k], color='k', linestyle='-.', label=f"$w_{k}$ paper Laplace")
  plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("w")
plt.xlabel("Iterations")
plt.title("sKF weights: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
# plt.savefig("figures/skf_weights.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
h_num   = MC_measures["sKF_integral"]['h']
h_paper = MC_measures["sKF_paper"]['h']
L = h_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(10*np.log10(np.abs(h_num[:, k] - h_paper[:,k])/np.abs(h_paper[:,k])),   color=c, linestyle='-',  label=f"$w_{k}$ numerical")
  #plt.plot(h_paper[:, k], color=c, linestyle='--', label=f"$w_{k}$ paper")
  plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("weights difference (numerical-paper)/paper [dB]")
plt.xlabel("Iterations")
plt.title("sKF weights: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
v_num   = MC_measures["sKF_integral"]['var']
v_paper = MC_measures["sKF_paper"]['var']
L = v_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(9, 5))
# for k in range(L):
#   c = colors[k % len(colors)]
#   plt.plot(v_num[:, k],   color=c, linestyle='-',  label=f"$v_{k}$ numerical")
#   plt.plot(v_paper[:, k], color=c, linestyle='--', label=f"$v_{k}$ paper")
  # plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot(v_num[:, 0],   color=c, linestyle='-',  label=f"v numerical")
plt.plot(v_paper[:, 0], color=c, linestyle='--', label=f"v paper")

#plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("v")
plt.xlabel("Iterations")
plt.title("sKF variance: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
v_num   = MC_measures["sKF_integral"]['var']
v_paper = MC_measures["sKF_paper"]['var']
L = v_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(9, 5))
# for k in range(L):
#   c = colors[k % len(colors)]
#   plt.plot(v_num[:, k],   color=c, linestyle='-',  label=f"$v_{k}$ numerical")
#   plt.plot(v_paper[:, k], color=c, linestyle='--', label=f"$v_{k}$ paper")
  # plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

# plt.plot(v_num[:, 0],   color=c, linestyle='-',  label=f"v numerical")
# plt.plot(v_paper[:, 0], color=c, linestyle='--', label=f"v paper")

plt.plot(10*np.log10(np.abs(v_num[:, 0] - v_paper[:, 0])/np.abs(v_paper[:, 0])), color=c, linestyle='--', label=f"v paper")

#plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("(v_num-v_paper)/v_paper [dB]")
plt.xlabel("Iterations")
plt.title("sKF variance difference: numerical integration vs paper recursion LOG")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
plt.show()

##### Laplacian Likelihood

In [ ]:
L = 3
ho = np.sinc(np.linspace(0, 1.5, L))
ho = ho / np.linalg.norm(ho)
h0 = np.zeros(L)
var_x = 1
var_v = 1e-3
# AR = np.array([1.0, 0.0])
AR = np.array([1.0, -0.6, 0.85])
# AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

Rxx_est = autocorr_matrix_estimate(x, M=L)
eig = np.linalg.eigvals(Rxx)
chi = np.max(eig) / np.min(eig)
print(f"Eigenvalue spread: {chi:.4} \n")

In [ ]:
NR = 1
N = int(200)

epsilon = 0.01
var_theta_0 = 2  # v_tilde_0
b_eta = 5 * np.sqrt(var_v)
dx_factor = 1/10
min_std_deviations = 3

Algorithms = [sKF_L_integral_algorithm, sKF_L_algorithm, sKF_L_exact_algorithm]

skf_L_parameters = np.void(
    ("sKF_L_integral", epsilon, var_theta_0, b_eta, dx_factor, min_std_deviations),
    dtype=skf_L_int_params,
)

sKF_L_parameters_minorized = np.void(
    ("sKF_L_minorized", epsilon, b_eta, var_theta_0), dtype=sKF_L_params
)

sKF_L_parameters_exact = np.void(
    ("sKF_L_exact", epsilon, b_eta, var_theta_0), dtype=sKF_L_params
)

Alg_Parameters = [skf_L_parameters, sKF_L_parameters_minorized, sKF_L_parameters_exact]

with ProgressBar(total=NR) as PBar:
    MC_measures = MC_Simulations_Modular_Variance(
        N, NR, ho, var_x, var_v, h0, Algorithms, Alg_Parameters, AR, PBar
    )

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(10 * np.log10(MC_measures["sKF_L_minorized"]["Jex"]))
plt.plot(10 * np.log10(MC_measures["sKF_L_exact"]["Jex"]))
plt.plot(10 * np.log10(MC_measures["sKF_L_integral"]["Jex"]))

plt.ylabel("EMSE (dB)")
plt.xlabel("Iterations")
plt.title("sKF integral Learning Curve")
plt.show()

In [ ]:
#h_num   = MC_measures["sKF_integral"]['h']
h_L_num   = MC_measures["sKF_L_integral"]['h']
#h_paper = MC_measures["sKF_paper"]['h']
h_L_minorized = MC_measures["sKF_L_minorized"]['h']
h_L_exact = MC_measures["sKF_L_exact"]['h']

L = h_L_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(h_L_num[:, k],   color=c, linestyle='-',  label=f"$w_{k}$ numerical")
  plt.plot(h_L_minorized[:, k], color=c, linestyle='--', linewidth=1, label=f"$w_{k}$ minorized")
  plt.plot(h_L_exact[:, k], color=c, linestyle='-.', label=f"$w_{k}$ exact")
  plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("w")
plt.xlabel("Iterations")
plt.title("sKF-L weights: numerical integration vs paper recursion")
# plt.legend(ncol=L, fontsize=9)
# --- PLOTTING THE PARAMETERS IN THE LEGEND ---
ax = plt.gca()
leg1 = ax.legend(ncol=L, fontsize=9, loc="upper right")
ax.add_artist(leg1)
def fmt(p):
    return "\n".join(f"{k} = {p[k]}" for k in p.dtype.names)
ax.legend(handles=[],
          title=f"integral:\n{fmt(skf_L_parameters)}\n\nminorized and exact:\n{fmt(sKF_L_parameters_minorized)}",
          loc="lower right", fontsize=8, title_fontsize=8)
# --- ---
plt.grid(alpha=0.3)
plt.savefig("skf_weights.png", dpi=300, bbox_inches="tight")
plt.show()


